# from app reviews to themes
Rob Held · Quant UX Con 2026 · [robertforrestheld.com](https://www.robertforrestheld.com)

The pipeline behind the talk, small enough to run on a laptop:
reviews → taxonomy → training data → classifier → theme shares → dashboard.

Run it from this folder. Needs pandas, scikit-learn, joblib (and matplotlib if you want plots).

### setup

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score

HERE = Path.cwd()                     #the repo folder
DATA = HERE / "data"
ACQ = pd.Timestamp("2025-03-01")      #bending spoons acquisition

#### 1. the reviews
10,000 German reviews of komoot and 9 competitors, Jan 2022 onwards, Play Store and App Store.
No scraping here. Original ids are replaced, and emails, links and @names are taken out of the text.

In [2]:
df = pd.read_csv(DATA / "reviews_sample.csv", parse_dates=["date"])
print(len(df), "reviews")

#how many per app, and the average rating
df.groupby("app")["rating"].agg(["mean", "count"]).round(2).sort_values("count", ascending=False)

10000 reviews


,mean,count
app,,
komoot,3.84,6766
mapycz,4.35,735
strava,2.66,606
outdooractive,3.19,446
bergfex,3.90,421
osmand,2.83,325
bikemap,1.94,244
alltrails,2.68,190
locusmap,3.33,138


#### 2. the taxonomy
What the labels mean: 20 themes, each with subthemes, a definition and a German example.
Built by hand from 1,500 reviews over four versions.
Note: `feature_loss` was later split, what's left of it is called `feature_removed` in the model.

In [3]:
tax = pd.read_csv(DATA / "taxonomy.csv")
print(tax.theme.nunique(), "themes ·", len(tax), "rows")

#one line per theme, its subthemes joined
tax.groupby("theme")["subtheme"].agg(lambda s: ", ".join(x for x in s.dropna() if x != "-"))

20 themes · 69 rows


theme
account_data                                                data_loss, login
app_update                                 ui_relearning, version_regression
community                            community_size, social_graph, ugc_tours
device_platform                               battery, gps_signal, os_compat
dissatisfaction_general                                                     
entitlement_violation      access_revoked, lifetime_purchase_ignored, pay...
exit_intent                announced_departure, conditional_threat, unins...
feature_loss                                               degraded, removed
feature_request                                                             
integrations               bike_computer, cross_device, ebike_system, sen...
other                                                  competitor_preference
positive_value             continuous_improvement, enables_delight, help_...
price                      dark_pattern, freemium_tier, subscription_m

#### 3. the training data
4,301 labelled reviews. Most were labelled by an LLM and then checked by hand where it was likely wrong,
the rest were labelled by hand to boost rare themes.

- `themes`: the labels, `|`-separated
- `labelled_for`: some rows were only checked for a few themes, so they only count for those
- `weight`: rows picked on purpose count half, so rare themes don't look more common than they are

`gold_heldout.csv` is 198 reviews labelled by hand and never used for training. It's only for scoring at the end.

In [4]:
train = pd.read_csv(DATA / "training_data.csv")
gold = pd.read_csv(DATA / "gold_heldout.csv")
for d in (train, gold):
    d["themes"] = d["themes"].fillna("").str.split("|")
train["labelled_for"] = train["labelled_for"].str.split("|")

#how many examples per theme
pd.Series([t for ts in train.themes for t in ts if t]).value_counts()

product                    1978
positive_value              953
price                       355
app_update                  211
feature_request             207
integrations                172
exit_intent                 172
community                   143
device_platform             141
entitlement_violation       139
upsell_pressure             136
feature_removed             116
account_data                110
other                        96
support                      95
trust_brand                  79
subscription_ops             76
privacy_data                 46
dissatisfaction_general      12
switching_barrier             7
Name: count, dtype: int64

#### 4. train the classifier
One small model per theme (a review can have several themes).
Themes with fewer than 30 examples are left out, they stay LLM-only.
Takes a couple of minutes.

In [5]:
meta = json.load(open(HERE / "models" / "theme_classifier.json"))
THEMES = meta["themes"]

#words + character pieces. the character part catches German compounds like akkuverbrauch or routenplanung
vec = FeatureUnion([
    ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                             strip_accents="unicode", max_features=60000)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True,
                             max_features=120000))])
X = vec.fit_transform(train.text.fillna(""))

models = {}
for t in THEMES:
    rows = train.labelled_for.apply(lambda k: "all" in k or t in k).values   #only rows checked for this theme
    y = train.themes.apply(lambda ts: t in ts).values
    m = CalibratedClassifierCV(LinearSVC(C=0.5, class_weight="balanced", random_state=42), cv=3)
    models[t] = m.fit(X[rows], y[rows], sample_weight=train.weight.values[rows])
print("trained", len(models), "themes")

#or skip the training and load the one used in the talk:
#import joblib; M = joblib.load(HERE / "models" / "theme_classifier.joblib"); vec, models = M["vectoriser"], M["models"]

trained 18 themes


#### 5. how good is it
Scored once on the held-out set. For comparison, the LLM got a micro-F1 of 0.757 on the same reviews.
Macro-F1 isn't shown on purpose: some themes have 1–3 examples here, so it swings with one prediction.

In [6]:
def predict(texts, cut):
    Xt = vec.transform(texts.fillna(""))
    P = pd.DataFrame({t: models[t].predict_proba(Xt)[:, 1] for t in THEMES})
    return P, (P >= pd.Series(cut)).astype(int)

truth = pd.DataFrame({t: gold.themes.apply(lambda ts: t in ts) for t in THEMES}).astype(int)
_, guess = predict(gold.text, meta["thresholds"])
print("micro-F1", round(f1_score(truth, guess, average="micro"), 3))

#per theme, with how many examples it had
pd.DataFrame({"n": truth.sum(), "f1": f1_score(truth, guess, average=None, zero_division=0).round(2)}).sort_values("n", ascending=False)

micro-F1 0.743


,n,f1
product,118,0.85
positive_value,47,0.73
integrations,17,0.71
price,16,0.77
feature_request,14,0.62
app_update,11,0.84
device_platform,10,0.60
community,9,0.78
upsell_pressure,7,0.33
other,6,0.33


#### 6. classify the sample
Two sets of cut-offs come with the model:
`thresholds` gives the best guess per review (good for picking quotes), `count_thresholds` makes totals match the hand labels (good for shares).
For shares we want the second one.

In [7]:
P, labels = predict(df.text, meta["count_thresholds"])
df = pd.concat([df.reset_index(drop=True), labels, P.add_prefix("p_").round(4)], axis=1)
df.to_csv(DATA / "classified_sample.csv", index=False)

#% of reviews per theme, komoot vs the other apps
df["group"] = np.where(df.app == "komoot", "komoot", "other apps")
share = df.groupby("group")[THEMES].mean().T.mul(100).round(1)
share["diff"] = share["komoot"] - share["other apps"]
share.sort_values("komoot", ascending=False)

group,komoot,other apps,diff
product,64.8,57.1,7.7
positive_value,26.8,18.6,8.2
price,6.7,15.8,-9.1
feature_request,6.1,6.9,-0.8
integrations,5.7,7.5,-1.8
device_platform,4.7,4.2,0.5
app_update,4.5,4.2,0.3
upsell_pressure,4.4,4.7,-0.3
community,2.9,1.6,1.3
entitlement_violation,2.8,3.0,-0.2


In [8]:
#komoot before vs after the acquisition
k = df[df.app == "komoot"]
pp = k.groupby(np.where(k.date >= ACQ, "after", "before"))[THEMES].mean().T.mul(100).round(1)[["before", "after"]]
pp.sort_values("after", ascending=False)

,before,after
product,65.0,64.4
positive_value,31.2,17.2
app_update,1.7,10.7
upsell_pressure,1.6,10.4
price,5.6,9.1
integrations,5.3,6.5
entitlement_violation,1.4,5.8
device_platform,4.2,5.7
trust_brand,0.1,5.6
exit_intent,1.4,5.2


In [9]:
#what a theme does to the rating: mean stars with the theme minus without it (an association, not an effect)
gap = pd.Series({t: k.loc[k[t] == 1, "rating"].mean() - k.loc[k[t] == 0, "rating"].mean() for t in THEMES})
pd.DataFrame({"share %": k[THEMES].mean().mul(100), "gap": gap}).round(2).sort_values("gap")

,share %,gap
privacy_data,0.58,-2.55
subscription_ops,0.50,-2.29
exit_intent,2.62,-2.04
entitlement_violation,2.76,-1.93
account_data,1.02,-1.92
trust_brand,1.88,-1.90
upsell_pressure,4.43,-1.88
feature_removed,0.90,-1.56
app_update,4.54,-1.54
price,6.70,-1.27


#### 7. the dashboard
`dashboard/index.html` opens in any browser, no server needed.
It shows the version built from all German reviews. This rebuilds it from the sample you just classified,
reload the page afterwards. Delete `my_dashboard_data.js` to go back.

In [10]:
%run dashboard/build_dashboard_data.py --src data/classified_sample.csv --out dashboard/data/my_dashboard_data.js

wrote dashboard/data/my_dashboard_data.js (88 kB) · through 2026-09-15 · current 2026-08 · partial=True
